# Notebook 1: Exploratory Data Analysis (EDA)

**Project:** Financial Fraud Detection  

**Goal:**  
- Load and understand the synthetic bank transaction dataset  
- Explore fraud distribution and class imbalance  
- Identify relationships between features and the 'isFraud' target  
- Generate hypotheses to guide preprocessing and modeling

In [4]:
#Import packages and Settings 

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")
pd.set_option("display.max_columns", None)

In [5]:
#Load Dataset - taking the transaction dataset from the data/ folder

df = pd.read_csv(".../Data/fraud_dataset.csv")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '.../Data/fraud_dataset.csv'

In [ ]:
df.shape, df.columns

In [ ]:
#Inspecting data types, non-null counts, and statistical summaries

df.info()

In [ ]:
df.describe(include="all")

#we want to include all numeric and non numeric columns 

In [ ]:
df["type"].value_counts()

In [ ]:
#Checking for missing values in the dataset. Very rare in this type of dataset, but still worth checking.

df.isnull().sum()

#none exist in our dataset

Before exploring relationships, we examine how imbalanced the *isFraud* target is.  
This helps us understand the difficulty of detecting fraud and why accuracy alone is misleading.

In [ ]:
fraud_counts = df["isFraud"].value_counts()
fraud_percent = df["isFraud"].mean() * 100

print("Fraud counts:\n", fraud_counts)
print(f"\nFraud percentage: {fraud_percent:.6f}%")

We examined so far:
- *type* — only 5 unique transaction types  
- *isFlaggedFraud* — a lofty rule that flags only extremely large transactions.  
- *nameOrig* and *nameDest* — extremely highly unique values in the ID fields (not useful for modeling)

**Univariate analysis — numeric features**

In [ ]:
# Univariate analysis — numeric features

#We explore distributions of key numeric variables.  
#These columns are heavily skewed, with many zeros and extremely large outliers — typical of financial data.
#Numeric columns examined:'step', 'amount' ,'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest'

numeric_cols = [
    "step",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
]

df[numeric_cols].hist(bins=50, figsize=(12, 8))
plt.tight_layout()
plt.show()

*Results*

**Step:**  
Most transactions happen between steps 0–400. This just means most activity in the dataset occurs in the earlier hours of the simulation.

**Amount:**  
Almost all transactions involve small amounts, but there are a few extremely large ones. This creates a long tail — most values near zero, a tiny number way out to the right.

**Old Balance (Origin):**  
Most sending accounts start with very little money (often zero). A small number of accounts have very large balances, which is why the graph stretches so far to the right.

**New Balance (Origin):**  
Same pattern — most accounts end with zero or close to zero after the transaction. This is common in fraud cases where the account gets drained.

**Old Balance (Destination):**  
Most receiving accounts also start with zero. A few accounts receive very large amounts, creating another long tail.

**New Balance (Destination):**  
Again, most accounts end with small balances, but some spike to very high values after receiving money.

**Overall takeaway:**  
All of these features are extremely skewed — lots of tiny values and a few huge ones. This helps explain why fraud often involves sudden, dramatic balance changes.

In [ ]:
#Value Counts and Plots
print("Transaction type counts:")
print(df["type"].value_counts())

sns.countplot(x="type", data=df, order=df["type"].value_counts().index)
plt.title("Transaction Types")
plt.xticks(rotation=45)
plt.show()

In [ ]:
print("\nisFlaggedFraud counts:")
print(df["isFlaggedFraud"].value_counts())

sns.countplot(x="isFlaggedFraud", data=df)
plt.title("isFlaggedFraud Distribution (Lofty Rule)")
plt.show()

**Bivariate Analysis**

In [ ]:
#**Bivariate analysis — "type" vs "isFraud"**
##Fraud is known to be concentrated in specific transaction types  
##(e.g., **TRANSFER** and **CASH_OUT**).  
##We compute fraud rates by type.

#Fraud rates by Type
fraud_by_type = df.groupby("type")["isFraud"].mean().sort_values(ascending=False)
print(fraud_by_type)

sns.barplot(x=fraud_by_type.index, y=fraud_by_type.values)
plt.title("Fraud Rate by Transaction Type")
plt.xlabel("Transaction Type")
plt.ylabel("Fraud Rate")
plt.xticks(rotation=45)
plt.show()

In [ ]:
#Bivariate analysis — amount vs `isFraud`
#We compare transaction amounts for fraudulent vs non-fraudulent transactions.

sns.boxplot(x="isFraud", y="amount", data=df)
plt.yscale("log")
plt.title("Transaction Amount by Fraud Label (Log Scale)")
plt.xlabel("isFraud")
plt.ylabel("Amount")
plt.show()

In [ ]:
#Bivariate analysis — origin balances vs `isFraud`
#We inspect how sender balances behave for fraudulent vs non-fraudulent transactions.
    ##Fraudulent transactions often: drain the origin account, show mismatches between amount and balance changes, involve accounts with previously low or zero balances.

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.boxplot(x="isFraud", y="oldbalanceOrg", data=df, ax=axes[0])
axes[0].set_title("Old Origin Balance by Fraud Label")
axes[0].set_yscale("log")

sns.boxplot(x="isFraud", y="newbalanceOrig", data=df, ax=axes[1])
axes[1].set_title("New Origin Balance by Fraud Label")
axes[1].set_yscale("log")

plt.tight_layout()
plt.show()